# Dependency

- vedo: 2024.5.2

# Common Libraries

In [2]:
import os, sys, glob
import numpy as np
from pathlib import Path
import matplotlib.pylab as plt
from skimage import io
from matplotlib.colors import to_hex
from vedo import settings
settings.default_backend = "vtk"

# Paths

In [ ]:
# afni path
abin_path = "/Users/seojin/abin"

# Directory path for saving cluster result
cluster_dir_path = "/Users/seojin/Desktop/MRI_visualization/Vedo_ROIs/brain_rois/clusters"

# Base brain mask - LPS+ coords(mni, spm)
base_brain_nii_path = "/Users/seojin/Desktop/MRI_visualization/Vedo_ROIs/group_mask.nii.gz"

# Brain roi path
brain_roi_path = "/Users/seojin/Desktop/MRI_visualization/Vedo_ROIs/brain_rois"

# Custom Libraries

In [ ]:
# Custom Libraries
from util.afni_extension import set_afni_abin, cluster_infos
from sj_brain_vis import show_clusterize_brain

set_afni_abin(abin_path)

# Parameters

In [5]:
# ROIs
roi_vtk_files = sorted(glob.glob(os.path.join(brain_roi_path, "*.vtk")))
roi_names = [Path(file).stem for file in roi_vtk_files]

In [6]:
brain_roi_path

'/Users/seojin/Desktop/MRI_visualization/Vedo_ROIs/brain_rois'

# Sort roi vtks

In [7]:
def get_roi_base_name(file_path):
    parts = Path(file_path).stem.split("_")
    return "_".join(parts[1:]) if len(parts) > 1 else parts[0]

# Sort roi files by base name (ignoring orientation prefix like Lt_/Rt_)
roi_vtk_files = sorted(roi_vtk_files, key=get_roi_base_name)
roi_names = [Path(file).stem for file in roi_vtk_files]
roi_name_noOrients = [get_roi_base_name(file) for file in roi_vtk_files]

# Take colormap
target_rois = ["precentral", "postcentral", "superior_parietal", "inferior_parietal"]
colors = plt.cm.tab10(np.linspace(0, 1, len(target_rois)))
colors = colors[::-1]

# Take color for mapping rois
color_i = 0
prev_roi_name = None
roi_colors = []
for roi_name in roi_name_noOrients:
    if prev_roi_name is not None and prev_roi_name != roi_name:
        color_i += 1

    try:
        roi_index = target_rois.index(roi_name)
        roi_colors.append(to_hex(colors[roi_index]))
    except:
        roi_colors.append("#929591")  # gray
    
    prev_roi_name = roi_name



In [8]:
roi_vtk_files = [path for path in roi_vtk_files if Path(path).stem.startswith("Rt")]

# Model

In [9]:
from collections.abc import Iterable
from pathlib import Path

import vedo
from vedo import Light, Plotter, Point, Text2D


def show_roi_vtk(
    roi_vtk_files,
    background_color="black",
    roi_style_info=None,
    is_custom_lightening=False,
    lightening_style_info=None,
    axis_info=None,
):
    """
    Visualize ROI VTK files without statistical maps or clusters.

    :param roi_vtk_files: Paths to ROI VTK files.
        Example: ["/path/to/roi_a.vtk", "/path/to/roi_b.vtk"]
    :param background_color: Background color of the visualization window.
        Example: "black" or "#000000"
    :param roi_style_info: Visualization settings for the ROI meshes.
        Supported keys are colors, opacities, lightenings, line_widths,
        line_colors, and adjust_methods.
    :param is_custom_lightening: Whether to enable interactive custom lighting.
    :param lightening_style_info: Custom lighting settings.
        Supported keys are radius, sphere_color, and lightening_color.
    :param axis_info: Axis mirroring settings.
        Supported keys are is_mirror_x, is_mirror_y, and is_mirror_z.
    """
    if roi_style_info is None:
        roi_style_info = {}

    if lightening_style_info is None:
        lightening_style_info = {}

    if axis_info is None:
        axis_info = {}

    if not roi_vtk_files:
        raise ValueError("roi_vtk_files must contain at least one VTK file.")

    def get_style_value(value, index):
        """
        Return the style value corresponding to the ROI index.

        :param value: A single style value or a sequence of style values.
        :param index: ROI index used to select a value from a sequence.
        :return: The selected style value.
        """
        if isinstance(value, Iterable) and not isinstance(value, str):
            values = list(value)

            if index >= len(values):
                raise IndexError(
                    "The number of style values is smaller than the number "
                    f"of ROI files. ROI index: {index}, "
                    f"number of style values: {len(values)}"
                )

            return values[index]

        return value

    roi_colors = roi_style_info.get("colors", "#929591")
    roi_opacities = roi_style_info.get("opacities", 0.1)
    roi_lightenings = roi_style_info.get("lightenings", "glossy")
    roi_line_widths = roi_style_info.get("line_widths", None)
    roi_line_colors = roi_style_info.get("line_colors", None)
    roi_adjusts = roi_style_info.get("adjust_methods", [])

    roi_vtk_volumes = []

    for roi_index, vtk_path in enumerate(roi_vtk_files):
        roi_vtk_volume = vedo.load(vtk_path)
        roi_vtk_volume.name = Path(vtk_path).stem

        roi_color = get_style_value(roi_colors, roi_index)
        roi_vtk_volume.color(roi_color)

        roi_opacity = get_style_value(roi_opacities, roi_index)
        roi_vtk_volume.opacity(roi_opacity)

        roi_lightening = get_style_value(roi_lightenings, roi_index)
        roi_vtk_volume.lighting(roi_lightening)

        if roi_line_widths is not None:
            roi_line_width = get_style_value(
                roi_line_widths,
                roi_index,
            )
            roi_vtk_volume.lw(roi_line_width)

        if roi_line_colors is not None:
            roi_line_color = get_style_value(
                roi_line_colors,
                roi_index,
            )
            roi_vtk_volume.lc(roi_line_color)

        for adjust_type, adjust_options in roi_adjusts:
            if adjust_options is None:
                adjust_options = {}

            if adjust_type == "subdivide":
                method = adjust_options.get("method", 0)
                mel = adjust_options.get("mel", None)

                roi_vtk_volume = roi_vtk_volume.subdivide(
                    method=method,
                    mel=mel,
                )

            elif adjust_type == "smooth":
                niter = adjust_options.get("niter", 15)
                boundary = adjust_options.get("boundary", False)

                roi_vtk_volume = roi_vtk_volume.smooth(
                    niter=niter,
                    boundary=boundary,
                )

            elif adjust_type == "clean":
                roi_vtk_volume = roi_vtk_volume.clean()

            elif adjust_type in {"is_normal", "compute_normals"}:
                points = adjust_options.get("points", True)
                cells = adjust_options.get("cells", True)
                consistency = adjust_options.get(
                    "consistency",
                    True,
                )

                roi_vtk_volume = roi_vtk_volume.compute_normals(
                    points=points,
                    cells=cells,
                    consistency=consistency,
                )

            elif adjust_type == "decimate":
                fraction = adjust_options.get("fraction", 0.5)
                method = adjust_options.get("method", "quadric")
                boundaries = adjust_options.get(
                    "boundaries",
                    False,
                )

                roi_vtk_volume = roi_vtk_volume.decimate(
                    fraction=fraction,
                    method=method,
                    boundaries=boundaries,
                )

            else:
                raise ValueError(
                    f"Unsupported ROI adjustment method: {adjust_type}"
                )

        roi_vtk_volume.pickable(False)
        roi_vtk_volumes.append(roi_vtk_volume)

    visualization_objects = []

    for roi_vtk_volume in roi_vtk_volumes:
        current_object = roi_vtk_volume

        if axis_info.get("is_mirror_x", False):
            current_object = current_object.clone().mirror("x")

        if axis_info.get("is_mirror_y", False):
            current_object = current_object.clone().mirror("y")

        if axis_info.get("is_mirror_z", False):
            current_object = current_object.clone().mirror("z")

        visualization_objects.append(current_object)

    control_info = {
        "mode": "normal",
        "target": None,
    }

    message = Text2D(
        "",
        pos="bottom-center",
        c="k",
        bg="r9",
        alpha=0.8,
    )

    auxiliary_message = Text2D(
        "",
        pos="top-left",
        c="k",
        bg="r9",
        alpha=0.8,
    )

    plotter = Plotter(
        axes=2,
        bg=background_color,
    )

    first_light = None
    first_light_point = None
    light_mapping = {}

    if is_custom_lightening:
        light_sphere_radius = lightening_style_info.get(
            "radius",
            15,
        )
        light_sphere_color = lightening_style_info.get(
            "sphere_color",
            "red",
        )
        light_color = lightening_style_info.get(
            "lightening_color",
            "white",
        )

        first_light_position = [0, 0, 0]

        first_light_point = (
            Point(r=light_sphere_radius)
            .pos(first_light_position)
            .c(light_sphere_color)
        )
        first_light_point.name = "1"

        first_light = Light(
            first_light_position,
            c=light_color,
            intensity=1,
        )
        first_light.name = "Light1"

        light_mapping[first_light_point.name] = first_light

    def handle_light_mouse_click(event):
        """
        Handle mouse interaction for custom lights.

        :param event: Vedo mouse callback event.
        """
        if not is_custom_lightening:
            return

        current_mode = control_info["mode"]

        if current_mode == "normal":
            return

        if current_mode == "select_light":
            if event.actor is None:
                return

            if event.actor.name not in light_mapping:
                return

            control_info["target"] = event.actor

            auxiliary_message.text(
                f"Selected light: {control_info['target'].name}"
            )
            plotter.render()

        elif current_mode == "move_light":
            target_point = control_info["target"]

            if target_point is None:
                return

            target_light = light_mapping.get(target_point.name)

            if target_light is None:
                return

            mouse_position_3d = plotter.compute_world_position(
                event.picked2d
            )

            target_point.pos(mouse_position_3d)
            target_light.SetPosition(mouse_position_3d)

            plotter.render()

        elif current_mode == "make_light":
            if light_mapping:
                point_number = (
                    max(int(name) for name in light_mapping.keys()) + 1
                )
            else:
                point_number = 1

            mouse_position_3d = plotter.compute_world_position(
                event.picked2d
            )

            new_light_point = (
                Point(r=light_sphere_radius)
                .pos(mouse_position_3d)
                .c(light_sphere_color)
            )
            new_light_point.name = str(point_number)

            new_light = Light(
                mouse_position_3d,
                c=light_color,
                intensity=1,
            )
            new_light.name = f"Light{point_number}"

            light_mapping[new_light_point.name] = new_light

            plotter.add(new_light_point, new_light)
            plotter.render()

    def handle_mouse_click(event):
        """
        Handle ROI selection and custom-light interaction.

        :param event: Vedo mouse callback event.
        """
        handle_light_mouse_click(event)

        if control_info["mode"] != "normal":
            return

        if event.actor is None:
            return

        actor_name = getattr(event.actor, "name", "")

        if actor_name:
            message.text(f"ROI: {actor_name}")
            plotter.render()

    def handle_key_press(event):
        """
        Change the custom-light interaction mode.

        :param event: Vedo keyboard callback event.
        """
        if not is_custom_lightening:
            return

        pressed_key = getattr(event, "key_pressed", None)

        if pressed_key is None and isinstance(event, dict):
            pressed_key = event.get("keyPressed")

        if pressed_key not in {"backslash", "\\"}:
            return

        mode_order = [
            "normal",
            "select_light",
            "move_light",
            "make_light",
        ]

        current_mode_index = mode_order.index(
            control_info["mode"]
        )
        next_mode_index = (
            current_mode_index + 1
        ) % len(mode_order)

        control_info["mode"] = mode_order[next_mode_index]

        auxiliary_message.text(
            f"Mode: {control_info['mode']}"
        )
        plotter.render()

    plotter.add_callback(
        "mouse click",
        handle_mouse_click,
    )
    plotter.add_callback(
        "key press",
        handle_key_press,
    )

    objects_to_show = visualization_objects + [
        message,
        auxiliary_message,
    ]

    if is_custom_lightening:
        objects_to_show.extend(
            [
                first_light,
                first_light_point,
            ]
        )

    plotter.show(
        *objects_to_show,
        zoom=1.2,
    )

    plotter.close()

In [ ]:
show_roi_vtk(
    roi_vtk_files=roi_vtk_files,
    background_color="white",
    roi_style_info={
        "colors": roi_colors,
        "opacities": np.repeat(0.5, len(roi_colors)),
        "lightenings": "glossy",
        "line_widths": 1,
        "line_colors": "black",
        "adjust_methods": [
            (
                "smooth",
                {
                    "niter": 15,
                    "boundary": False,
                },
            ),
        ],
    },
    axis_info={
        "is_mirror_x": False,
        "is_mirror_y": False,
        "is_mirror_z": False,
    },
)

: 

In [13]:
vedo.__version__

'2024.5.2'